In [55]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [86]:
import pandas as pd
from datetime import datetime, timedelta
import urllib.parse
import pytz
from collections import defaultdict

# --- ThreatConnect setup (assumes 'tc' and 'ro' objects already exist) ---
# Make sure tc and ro are defined in your environment based on your SDK usage

# Define the time window (3 days ago at 00:00 UTC)
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Indicator types to query
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"
]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

# Owners to pull from
list_of_owners = ['HTOC Org']
final_results = []

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean the indicators
if final_results:
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')

        # Ensure lastObserved is a datetime object for filtering
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
        start_dt = pd.to_datetime(start)
        observed_src = observed_src[observed_src['lastObserved'] >= start_dt]

        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")

        # -------------------------------
        # Enrich indicators with VirusTotalV3
        # -------------------------------
        indicator_ids = observed_src['id'].dropna().unique().tolist()
        enriched_results = []

        print(f"\nEnriching {len(indicator_ids)} indicators with VirusTotalV3...")

        for indicator_id in indicator_ids:
            try:
                enrich_url = f'/v3/indicators/{indicator_id}/enrich?type=VirusTotalV3'
                ro.set_http_method('POST')
                ro.set_request_uri(enrich_url)
                ro.set_body({})  # Enrichment call usually has no body

                enrich_response = tc.api_request(ro)

                if enrich_response.status_code == 200:
                    enrich_data = enrich_response.json()
                    enrich_data['indicator_id'] = indicator_id
                    enriched_results.append(enrich_data)

            except Exception as e:
                continue

        # Convert enriched data into DataFrame
        if enriched_results:
            df_enriched = pd.json_normalize(enriched_results)
            df_enriched.rename(columns={'indicator_id': 'id'}, inplace=True)

            # Merge enrichment data into main dataframe
            observed_src = observed_src.merge(df_enriched, on='id', how='left')
            print(f"\nSuccessfully enriched and merged {len(df_enriched)} indicators.")
        else:
            print("\nNo enrichment data retrieved.")

    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 941 unique observed indicators.

Enriching 941 indicators with VirusTotalV3...


Status Code: 400
Failed API Response: b'{"message":"Unable to enrich this indicator. Either the indicator or enrichment types are not compatible, or some other kind of error occurred while trying to get the enrichment information.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL denverite.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL chaturbate.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL fmovies.co cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The Stripped URL moplay.org cannot be enriched with VirusTotal because the


Successfully enriched and merged 919 indicators.


In [87]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,description,...,data.privateFlag,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive,data.source
0,5629499555060730,2025-06-16T17:42:54Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:50:52Z,3.0,80,RITM0589984,...,False,True,False,5.188.206.66,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 7}]",NaN,NaN,NaN,NaN
1,6755399459033757,2025-06-16T17:42:40Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:50:49Z,3.0,80,RITM0589984,...,False,True,False,179.43.159.200,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 9}]",NaN,NaN,NaN,NaN
2,6755399459033760,2025-06-16T17:42:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:50:29Z,3.0,80,RITM0589984,...,False,True,False,109.70.100.2,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 11}]",NaN,NaN,NaN,NaN
3,5629499555060720,2025-06-16T17:42:49Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:02:29Z,3.0,80,RITM0589984,...,False,True,False,45.138.16.230,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 13}]",NaN,NaN,NaN,NaN
4,5629499555060740,2025-06-16T17:42:59Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-06-18T12:02:07Z,3.0,80,RITM0589984,...,False,True,False,94.16.115.121,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 5}]",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
936,4324196,2023-04-21T14:22:00Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-16T23:23:57Z,3.0,84,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
937,4585198,2024-04-30T15:02:57Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-04-30T15:24:35Z,4.0,51,VA users received email messages from <Good Ch...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
938,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
939,4512619,2024-01-26T20:41:51Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-01-28T12:55:28Z,3.0,70,"On December 28, 2023, a VA user received and r...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=5)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250618.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250617.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250616.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250615.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250614.csv']
Loaded data from 5 files.


In [90]:
import pandas as pd
from datetime import timedelta
from pandas import Timestamp

# Setup cutoff timestamps
cutoff = Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize filtered tags DataFrame
filtered_tags = pd.DataFrame()

# Filter API tags and attach metadata
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')

    if isinstance(tags_data, list):
        tags = pd.json_normalize(tags_data)
        tags['name'] = tags['name'].astype(str)
        all_tags_list = tags['name'].tolist()

        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()
        if not api_tags.empty:
            metadata_cols = [
                'summary', 'observations', 'description', 'type',
                'dateAdded', 'lastModified', 'lastObserved', 'webLink', 'data.enrichment.data'
            ]
            for col in metadata_cols:
                value = row.get(col)
                # If value is a list and not a string, assign [value] * len(api_tags)
                if isinstance(value, list) and not isinstance(value, str):
                    api_tags[col] = [value] * len(api_tags)
                else:
                    api_tags[col] = value
            # Fix: assign all_tags as a list of lists, one per row
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Convert date columns
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Validate required columns
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean 'name' column and match observations
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
filtered_tags['observed_date'] = pd.NaT

for index, row in filtered_tags.iterrows():
    match = observed_data_df[
        (observed_data_df['indicator'] == row['summary']) &
        (observed_data_df['OpDiv'] == row['cleaned_name'])
    ]
    if not match.empty:
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter recent tags (last 24 hours)
cutoff_naive = cutoff.tz_localize(None) if cutoff.tzinfo else cutoff
recent_tags = filtered_tags[
    (filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=1))
].copy()

# Extract partner from name and group
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Remove duplicate summaries
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    recent_tags = recent_tags[recent_tags['vtMaliciousCount'] > 10]
    
# Drop unused columns if present
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name', 'data.enrichment.data'
]
recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], inplace=True, errors='ignore')

# Optional: remove certain tags
# recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x or 'htoc_wl' in x if isinstance(x, list) else False)]

recent_tags

,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
1,30479,2025-06-18T12:02:29Z,109.70.100.2,58,RITM0589984,Address,2025-06-16 17:42:43+00:00,2025-06-18T12:50:29Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-18,1,CMS,11.0
15,35760,2025-06-18T12:50:29Z,78.153.140.177,3009,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:32+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, NIH Splunk API...",2025-06-18,3,"FDA, IHS, OS",12.0
33,471298,2025-06-18T10:27:50Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0
38,471298,2025-06-18T10:27:50Z,167.99.13.19,76504,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"DHA, FDA, HRSA, IHS, OS",12.0
64,471298,2025-06-18T10:27:50Z,178.128.84.112,18634,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:30+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,4,"DHA, FDA, HRSA, OS",11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1092,471298,2025-06-18T10:27:50Z,104.167.221.114,785920,TASK0882701 / RITM0585661,Address,2025-06-02 18:38:56+00:00,2025-06-16T11:25:13Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, HRSA, IHS, NIH, OS",19.0
1102,35760,2025-06-18T12:50:29Z,118.193.72.187,9802664,CMS Anomali ThreatStream Indicators from 02.14...,Address,2024-03-29 14:31:35+00:00,2025-06-16T11:25:13Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, cms-soc, OS Splunk API, VA OI...",2025-06-18,3,"CMS, HHS, OS",11.0
1110,35057,2025-06-18T08:50:27Z,183.224.237.233,297441,The following IP that made multiple attempts (...,Address,2025-01-24 20:59:00+00:00,2025-06-15T11:25:30Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Active Scanning, OS Splunk API, FDA Splunk AP...",2025-06-18,3,"CMS, FDA, HRSA",11.0
1123,30479,2025-06-18T12:02:29Z,185.220.101.182,223971,185.220.101[.]182 is registered in Germany und...,Address,2021-12-15 13:22:24+00:00,2025-06-14T11:22:40Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Zero Day, HHS Splunk Production API, HRSA Spl...",2025-06-18,1,CMS,12.0


In [91]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in recent_tags['partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: recent_tags[recent_tags['partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: FDA (32 records)


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
15,35760,2025-06-18T12:50:29Z,78.153.140.177,3009,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:32+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, NIH Splunk API...",2025-06-18,3,"FDA, IHS, OS",12.0
33,471298,2025-06-18T10:27:50Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0
38,471298,2025-06-18T10:27:50Z,167.99.13.19,76504,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"DHA, FDA, HRSA, IHS, OS",12.0
64,471298,2025-06-18T10:27:50Z,178.128.84.112,18634,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:30+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,4,"DHA, FDA, HRSA, OS",11.0
95,471298,2025-06-18T10:27:50Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",14.0
126,35760,2025-06-18T12:50:29Z,104.152.52.148,613,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:48+00:00,2025-06-18T11:22:49Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-06-18,2,"FDA, OS",11.0
163,35057,2025-06-18T08:50:27Z,104.152.52.216,665,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:43+00:00,2025-06-18T11:22:47Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-06-18,1,FDA,11.0
193,35760,2025-06-18T12:50:29Z,78.153.140.179,16448,RITM0581780/TASK0877793,Address,2025-05-19 11:39:42+00:00,2025-06-18T11:22:46Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-18,3,"FDA, IHS, OS",15.0
196,35057,2025-06-18T08:50:27Z,87.236.176.176,2199,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:42+00:00,2025-06-18T11:22:46Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-18,1,FDA,11.0
236,471298,2025-06-18T10:27:50Z,142.93.115.5,25450,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:22+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0


Partner: CMS (39 records)


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
1,30479,2025-06-18T12:02:29Z,109.70.100.2,58,RITM0589984,Address,2025-06-16 17:42:43+00:00,2025-06-18T12:50:29Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, CMS Splunk API, VA CSOC CTS Sp...",2025-06-18,1,CMS,11.0
33,471298,2025-06-18T10:27:50Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0
95,471298,2025-06-18T10:27:50Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",14.0
111,30479,2025-06-18T12:02:29Z,64.227.146.243,4111601,CMS Anomali ThreatStream Indicators from 01.14...,Address,2024-03-29 14:24:45+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[cms-soc, OS Splunk API, FDA Splunk API, CMS S...",2025-06-18,2,"CMS, HHS",12.0
229,30479,2025-06-18T12:02:29Z,185.220.100.240,1280292,Our partner Health-ISAC is sharing IOCs for Bl...,Address,2021-12-15 13:22:24+00:00,2025-06-18T11:22:45Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Amber Members, Incidents, Health-ISAC, OS Spl...",2025-06-18,1,CMS,14.0
236,471298,2025-06-18T10:27:50Z,142.93.115.5,25450,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:22+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0
322,35760,2025-06-18T12:50:29Z,80.67.167.81,28639,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-06-18T11:22:43Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[UAC-0082, Malware: ZeroWipe, Malware: CaddyWi...",2025-06-18,5,"CMS, FDA, HRSA, IHS, OS",15.0
335,471298,2025-06-18T10:27:50Z,185.169.4.150,2378665,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",20.0
366,471298,2025-06-18T10:27:50Z,152.32.128.214,3674721,NaN,Address,2024-07-08 15:36:44+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, FDA, HHS, HRSA, IHS",11.0
375,30479,2025-06-18T12:02:29Z,185.129.62.62,176065,TLP:WHITE\r\n\r\nJoint Analysis Report from NC...,Address,2017-01-03 15:27:15+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[CVE-2023-20198, Midnight Blizzard, Forest Bli...",2025-06-18,2,"CMS, HRSA",11.0


Partner: IHS (17 records)


...

,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
1047,35760,2025-06-18T12:50:29Z,78.153.140.224,17234,TASK0882701 / RITM0585661,Address,2025-06-02 18:33:32+00:00,2025-06-16T11:25:16Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-18,4,"FDA, IHS, NIH, OS",13.0
1092,471298,2025-06-18T10:27:50Z,104.167.221.114,785920,TASK0882701 / RITM0585661,Address,2025-06-02 18:38:56+00:00,2025-06-16T11:25:13Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, HRSA, IHS, NIH, OS",19.0


Partner: HRSA (36 records)


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
33,471298,2025-06-18T10:27:50Z,45.95.146.59,276688,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:31+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0
38,471298,2025-06-18T10:27:50Z,167.99.13.19,76504,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"DHA, FDA, HRSA, IHS, OS",12.0
64,471298,2025-06-18T10:27:50Z,178.128.84.112,18634,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:30+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,4,"DHA, FDA, HRSA, OS",11.0
95,471298,2025-06-18T10:27:50Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",14.0
236,471298,2025-06-18T10:27:50Z,142.93.115.5,25450,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:22+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"CMS, DHA, FDA, HRSA, IHS",12.0
322,35760,2025-06-18T12:50:29Z,80.67.167.81,28639,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-06-18T11:22:43Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[UAC-0082, Malware: ZeroWipe, Malware: CaddyWi...",2025-06-18,5,"CMS, FDA, HRSA, IHS, OS",15.0
335,471298,2025-06-18T10:27:50Z,185.169.4.150,2378665,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",20.0
344,35760,2025-06-18T12:50:29Z,178.20.55.16,12176,The following source IPs are related to resour...,Address,2017-01-03 15:27:15+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-06-18,2,"HRSA, OS",12.0
366,471298,2025-06-18T10:27:50Z,152.32.128.214,3674721,NaN,Address,2024-07-08 15:36:44+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, FDA, HHS, HRSA, IHS",11.0
375,30479,2025-06-18T12:02:29Z,185.129.62.62,176065,TLP:WHITE\r\n\r\nJoint Analysis Report from NC...,Address,2017-01-03 15:27:15+00:00,2025-06-18T11:22:41Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[CVE-2023-20198, Midnight Blizzard, Forest Bli...",2025-06-18,2,"CMS, HRSA",11.0


Partner: OS (40 records)


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
15,35760,2025-06-18T12:50:29Z,78.153.140.177,3009,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:32+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, NIH Splunk API...",2025-06-18,3,"FDA, IHS, OS",12.0
38,471298,2025-06-18T10:27:50Z,167.99.13.19,76504,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:27+00:00,2025-06-18T11:22:51Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,5,"DHA, FDA, HRSA, IHS, OS",12.0
64,471298,2025-06-18T10:27:50Z,178.128.84.112,18634,RITM0588707 / TASK0886419,Address,2025-06-11 14:46:30+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,4,"DHA, FDA, HRSA, OS",11.0
95,471298,2025-06-18T10:27:50Z,91.191.209.198,38425685,RITM0580258/TASK0875884,Address,2025-05-14 12:23:46+00:00,2025-06-18T11:22:50Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",14.0
126,35760,2025-06-18T12:50:29Z,104.152.52.148,613,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:48+00:00,2025-06-18T11:22:49Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-06-18,2,"FDA, OS",11.0
128,35760,2025-06-18T12:50:29Z,171.25.193.20,22240,TLP:WHITE\r\n\r\nJoint Analysis Report from NC...,Address,2017-01-03 15:27:15+00:00,2025-06-18T11:22:49Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Poseidon, Poemgate, CVE-2023-20198, Midnight ...",2025-06-18,1,OS,16.0
193,35760,2025-06-18T12:50:29Z,78.153.140.179,16448,RITM0581780/TASK0877793,Address,2025-05-19 11:39:42+00:00,2025-06-18T11:22:46Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-06-18,3,"FDA, IHS, OS",15.0
297,35760,2025-06-18T12:50:29Z,185.241.208.204,37628,**(U//FOUO) Additional Indicators of Compromis...,Address,2024-06-17 18:10:46+00:00,2025-06-18T11:22:44Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Cyber Intelligence Network (CIN), Utah, Energ...",2025-06-18,1,OS,13.0
322,35760,2025-06-18T12:50:29Z,80.67.167.81,28639,Executive Summary: \n\n80.67.167[.]81 resolve...,Address,2025-04-28 19:45:39+00:00,2025-06-18T11:22:43Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[UAC-0082, Malware: ZeroWipe, Malware: CaddyWi...",2025-06-18,5,"CMS, FDA, HRSA, IHS, OS",15.0
335,471298,2025-06-18T10:27:50Z,185.169.4.150,2378665,TASK0883886 / RITM0586633,Address,2025-06-04 16:43:18+00:00,2025-06-18T11:22:42Z,2025-06-18 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-06-18,6,"CMS, DHA, FDA, HRSA, IHS, OS",20.0


import os
import re
import time
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'

# Start Outlook
outlook = win32.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
drafts_folder = namespace.GetDefaultFolder(16)  # olFolderDrafts = 16

for partner, df in partner_buckets.items():
    try:
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)
        partner_folder = os.path.join(Tippers_Path, safe_partner)
        os.makedirs(partner_folder, exist_ok=True)

        # Save CSV to disk (Outlook needs a real file to attach)
        csv_filename = f"{safe_partner}_tippers.csv"
        csv_path = os.path.join(partner_folder, csv_filename)
        df.to_csv(csv_path, index=False)
        time.sleep(0.5)  # ensure write completes

        # Create mail item and save to Drafts
        mail = outlook.CreateItem(0)
        mail.Subject = f"Tippers Data for {partner}"
        mail.To = f"{safe_partner.lower()}@yourdomain.com"
        mail.Body = f"Attached is the Tippers CSV file for {partner}."
        mail.Attachments.Add(Source=csv_path)
        
        # Move to Drafts folder
        mail = mail.Move(drafts_folder)
        print(f"Draft saved in Outlook for: {partner}")

    except Exception as e:
        print(f"Error processing partner '{partner}': {e}")


In [97]:
import os
import re
import time
from datetime import datetime
import pandas as pd
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'
os.makedirs(Tippers_Path, exist_ok=True)

# Add today's date to the filename
today_str = datetime.utcnow().strftime("%Y%m%d")
excel_path = os.path.join(Tippers_Path, f"tippers_by_partner_{today_str}.xlsx")

with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    for partner, df in partner_buckets.items():
        # Excel sheet names can't be longer than 31 chars or contain special chars
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)[:31]

        # Convert all timezone-aware datetime columns to naive
        for col in df.select_dtypes(include=['datetimetz']).columns:
            df[col] = df[col].dt.tz_localize(None)

        # Write to Excel first
        df.to_excel(writer, sheet_name=safe_partner, index=False)

        # Then get the worksheet object to format
        worksheet = writer.sheets[safe_partner]
        worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        worksheet.freeze_panes(1, 0)

        for i, col in enumerate(df.columns):
            # Set width based on max text length in the column, or column name
            max_len = max(
                df[col].astype(str).map(len).max() if not df.empty else 0,
                len(str(col))
            )
            worksheet.set_column(i, i, min(max_len + 2, 50))  # Cap width at 50

print(f"Excel file with partner tabs saved to: {excel_path}")


Excel file with partner tabs saved to: Z:\HTOC\HTOC Reports\Tippers\tippers_by_partner_20250618.xlsx
